In [1]:
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_community.document_loaders import OnlinePDFLoader

In [ ]:
local_path = "1.pdf"

# Local PDF file uploads
if local_path:
  loader = UnstructuredPDFLoader(file_path=local_path)
  data = loader.load()
else:
  print("Upload a PDF file")

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("1.pdf")
pages = loader.load_and_split()

In [4]:
pages[0]

Document(metadata={'source': '1.pdf', 'page': 0}, page_content='77Ensuring Cross-Cultural Equivalence in \nTranslation of Research Consents and Clinical Documents\nA Systematic Process for Translating English to Chinese\nCheng-Chih Lee, MS, RN\nUniversity of California, San Francisco\nDenise Li, PhD, RN\nCalifornia State University, East Bay\nShoshana Arai, PhD, RN\nKathleen Puntillo, DNSc, RN, FAAN\nUniversity of California, San Francisco\nThe aim of this article is to describe a formal process used to translate research study materials from English into traditiona l\nChinese characters. This process may be useful for translating documents for use by both research participants and clinical\npatients. A modified Brislin model was used as the systematic translation process. Four bilingual translators were involved,and a Flaherty 3-point scale was used to evaluate the translated documents. The linguistic discrepancies that arise in theprocess of ensuring cross-cultural congruency or equi

In [3]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

In [4]:
# Split and chunk 
text_splitter = RecursiveCharacterTextSplitter(chunk_size=7500, chunk_overlap=100)
chunks = text_splitter.split_documents(pages)

In [5]:
# Add to vector database
vector_db = Chroma.from_documents(
    documents=chunks, 
    embedding=OllamaEmbeddings(model="nomic-embed-text",show_progress=True),
    collection_name="local-rag"
)

OllamaEmbeddings: 100%|██████████| 9/9 [00:56<00:00,  6.30s/it]


In [6]:
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_models import ChatOllama
from langchain_core.runnables import RunnablePassthrough
from langchain.retrievers.multi_query import MultiQueryRetriever

In [7]:
local_model = "mistral"
llm = ChatOllama(model=local_model)

In [8]:
QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""You are an AI language model assistant. Your task is to generate five
    different versions of the given user question to retrieve relevant documents from
    a vector database. By generating multiple perspectives on the user question, your
    goal is to help the user overcome some of the limitations of the distance-based
    similarity search. Provide these alternative questions separated by newlines.
    Original question: {question}""",
)

In [9]:
retriever = MultiQueryRetriever.from_llm(
    vector_db.as_retriever(), 
    llm,
    prompt=QUERY_PROMPT
)

# RAG prompt
template = """Answer the question based ONLY on the following context:
{context}
Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

In [10]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [11]:
chain.invoke(input("?"))

OllamaEmbeddings: 100%|██████████| 1/1 [00:02<00:00,  2.15s/it]


' The text discusses a cross-cultural translation process for scientific research, specifically studies conducted on Chinese populations. This process aims to achieve content, semantic, technical, criterion, or conceptual equivalence between the original and translated documents. The process involves multiple rounds of blinded back translations by independent bilingual translators to ensure accuracy and mutual agreement on meaning. The text also discusses some challenges encountered during translation, such as different grammatical and syntactical styles, cultural differences, and the importance of achieving semantic and conceptual equivalence rather than a literal translation that may result in distortions or incomplete sentences.'